# Solution: Time-Series Modeling with statsmodels

A regional electric utility needs to forecast monthly electricity demand 12 months ahead. You have 8 years of monthly data with demand and temperature. Fit ARIMA and SARIMAX models and generate forecasts with prediction intervals.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

HORIZON = 12


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
# Udacity brand colors
UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62", "Medium Gray": "#9C9FA8", "Gray": "#CECFD4", "Light Gray": "#F2F2F2", "White": "#FFFFFF"}
US = {"Orange": "#EE7622", "Yellow": "#F9DC5C", "Red": "#9C0D22", "Green": "#23CE6B"}# Thicker lines for visibility
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")

train_end = "2023-12-01"
train_df = df.loc[:train_end]
test_df = df.loc[train_end:].iloc[1:]

train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)

print(f"Data: {len(df)} months, {df.index[0].date()} to {df.index[-1].date()}")
print(f"Train: {len(train_df)} months, Test: {len(test_df)} months")


In [ ]:
axes[0].set_ylabel("Correlation", fontsize=16)
diffed = train_demand.diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=36)
plot_pacf(diffed, ax=axes[1], lags=36)
axes[0].set_title("ACF (seasonally differenced)", fontsize=18, fontweight="bold")
axes[0].tick_params(axis="both", labelsize=14)
axes[1].set_title("PACF (seasonally differenced)", fontsize=18, fontweight="bold")
axes[1].tick_params(axis="both", labelsize=14)
plt.tight_layout()
plt.show()

## Task 1: Fit ARIMA

Fit a SARIMAX model (no seasonal or exogenous components) with order `(2,1,1)`. Return the fitted result.

In [ ]:
def fit_arima(train, order=(2, 1, 1)):
    model = SARIMAX(train, order=order)
    return model.fit(disp=False)

arima_fit = fit_arima(train_demand)


## Task 2: Fit SARIMAX with Temperature

Fit a SARIMAX model with order `(1,1,1)`, seasonal_order `(1,1,1,12)`, and temperature as exogenous. Return the fitted result.

In [ ]:
def fit_sarimax(train, exog, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)):
    model = SARIMAX(train, exog=exog, order=order, seasonal_order=seasonal_order)
    return model.fit(disp=False)

sarimax_fit = fit_sarimax(train_demand, train_temp)


## Task 3: Forecast with Prediction Intervals

Use `fitted.get_forecast(steps, exog=exog_future)`. Return `(predicted_mean, conf_int DataFrame)`.

In [ ]:
def forecast_with_intervals(fitted, steps, exog_future=None, alpha=0.05):
    forecast = fitted.get_forecast(steps=steps, exog=exog_future)
    mean = forecast.predicted_mean
    ci = forecast.conf_int(alpha=alpha)
    return mean, ci

arima_fc, arima_ci = forecast_with_intervals(arima_fit, HORIZON)


In [ ]:
sarimax_fc, sarimax_ci = forecast_with_intervals(
    sarimax_fit, HORIZON, exog_future=future_temp.values
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
for ax, name, fc, ci, color in [
    (axes[0], "ARIMA(2,1,1)", arima_fc, arima_ci, UB["Brand Blue"]),
    (axes[1], "SARIMAX + temperature", sarimax_fc, sarimax_ci, US["Orange"]),
]:
    ax.plot(train_demand.index, train_demand.values, color=UN["Black"], label="Historical")
    ax.plot(fc.index, fc.values, color=color, linestyle="--", label="Forecast")
    ax.fill_between(fc.index, ci.iloc[:, 0], ci.iloc[:, 1], color=color, alpha=0.15, label="95% interval")
    ax.set_ylabel("Demand (MWh)", fontsize=16)
    ax.set_title(name, fontsize=18, fontweight="bold")
    ax.tick_params(axis="both", labelsize=14)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()